# <a id='toc1_'></a>[ENS Challenge #22 – Volatility Prediction](#toc0_)

This notebook downloads the data from Hugging Face, builds intraday features, runs extensive experiments (LightGBM primary, optional CatBoost/XGBoost), performs tuning with Optuna, and generates a submission file (`submission.csv`).

**Metric:** MAPE (Mean Absolute Percentage Error).


**Table of contents**<a id='toc0_'></a>    
- [ENS Challenge #22 – Volatility Prediction](#toc1_)    
  - [Download data from Hugging Face](#toc1_1_)    
  - [Load data](#toc1_2_)    
  - [Load target and merge](#toc1_3_)    
  - [Feature engineering](#toc1_4_)    
  - [Prepare datasets](#toc1_5_)    
  - [Evaluation metric (MAPE)](#toc1_6_)    
  - [Baselines](#toc1_7_)    
  - [Cross-validation setup](#toc1_8_)    
  - [LightGBM training & evaluation](#toc1_9_)    
  - [Hyperparameter tuning with Optuna (LightGBM)](#toc1_10_)    
  - [Optional models (CatBoost / XGBoost) and ensembling](#toc1_11_)    
  - [Final training and submission](#toc1_12_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

In [2]:
# Install dependencies (run once). If a package is missing, uncomment and run.
!pip3 install -U numpy pandas scikit-learn lightgbm optuna huggingface_hub pyarrow
# Optional:
!pip3 install -U catboost xgboost polars


In [3]:
import os
from pathlib import Path
import json
from datetime import datetime
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_percentage_error
from tqdm.auto import tqdm

RANDOM_SEED = 42
DATASET_ID = "Noptus/challenge_data_ens_22"
TIME_BUDGET_HOURS = 8
USE_GPU = False

# Detect Colab and set a suitable data directory
IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

DATA_DIR = Path("/content/data" if IN_COLAB else "data")
DATA_DIR.mkdir(exist_ok=True)

# Toggle quick vs full runs
RUN_MODE = "full"  # "quick" or "full"

# Run directory
RUN_DIR = Path("runs") / datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR.mkdir(parents=True, exist_ok=True)

run_config = {
    "random_seed": RANDOM_SEED,
    "dataset_id": DATASET_ID,
    "time_budget_hours": TIME_BUDGET_HOURS,
    "use_gpu": USE_GPU,
    "run_mode": RUN_MODE,
}
(RUN_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2))

np.random.seed(RANDOM_SEED)


/Users/raphaelcaillon/Documents/GitHub/challenge_data_ens_22/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## <a id='toc1_1_'></a>[Download data from Hugging Face](#toc0_)

In [4]:
from huggingface_hub import snapshot_download

allow_patterns = [
    "training_input.csv",
    "testing_input.csv",
    "challenge_34_cfm_trainingoutputfile (2).csv",
]

local_dir = snapshot_download(
    repo_id=DATASET_ID,
    repo_type="dataset",
    allow_patterns=allow_patterns,
    local_dir=DATA_DIR,
    local_dir_use_symlinks=False,
)
print("Downloaded to:", local_dir)

# Resolve file paths
train_input_path = DATA_DIR / "training_input.csv"
test_input_path = DATA_DIR / "testing_input.csv"
train_target_path = DATA_DIR / "challenge_34_cfm_trainingoutputfile (2).csv"

for p in [train_input_path, test_input_path, train_target_path]:
    print(p, p.exists(), p.stat().st_size if p.exists() else None)


/Users/raphaelcaillon/Documents/GitHub/challenge_data_ens_22/.venv/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Fetching 3 files: 100%|██████████| 3/3 [00:00<00:00, 2336.66it/s]

Downloaded to: /Users/raphaelcaillon/Documents/GitHub/challenge_data_ens_22/data
data/training_input.csv True 659997687
data/testing_input.csv True 658950276
data/challenge_34_cfm_trainingoutputfile (2).csv True 13979064


## <a id='toc1_2_'></a>[Load data](#toc0_)

In [5]:
def infer_col_groups(columns):
    vol_cols = [c for c in columns if c.startswith("volatility ")]
    ret_cols = [c for c in columns if c.startswith("return ")]
    return vol_cols, ret_cols

# Read header first to infer dtypes
train_cols = pd.read_csv(train_input_path, sep=";", nrows=0).columns
vol_cols, ret_cols = infer_col_groups(train_cols)

# dtype map
# ID/date/product_id as int32, vol as float32, return as float32
use_dtypes = {"ID": "int32", "date": "int32", "product_id": "int32"}
use_dtypes.update({c: "float32" for c in vol_cols})
use_dtypes.update({c: "float32" for c in ret_cols})

train_df = pd.read_csv(train_input_path, sep=";", dtype=use_dtypes)
test_df = pd.read_csv(test_input_path, sep=";", dtype=use_dtypes)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# Basic checks
assert train_df["ID"].is_unique
assert test_df["ID"].is_unique
assert len(vol_cols) > 0 and len(ret_cols) > 0
print("Vol cols:", len(vol_cols), "Return cols:", len(ret_cols))


Train shape: (636313, 111)
Test shape: (635397, 111)
Vol cols: 54 Return cols: 54


In [6]:
train_df.head()

,ID,date,product_id,volatility 09:30:00,volatility 09:35:00,volatility 09:40:00,volatility 09:45:00,volatility 09:50:00,volatility 09:55:00,volatility 10:00:00,...,return 13:10:00,return 13:15:00,return 13:20:00,return 13:25:00,return 13:30:00,return 13:35:00,return 13:40:00,return 13:45:00,return 13:50:00,return 13:55:00
0,1,1,1,0.662737,0.716896,0.698601,0.480172,0.623665,0.201876,0.327206,...,1.0,1.0,1.0,-1.0,1.0,-1.0,0.0,1.0,1.0,-1.0
1,2,1,2,0.285394,0.379560,0.094858,0.094794,0.055315,0.173994,0.102745,...,1.0,1.0,1.0,1.0,1.0,-1.0,-1.0,1.0,-1.0,-1.0
2,3,1,3,1.151646,1.093562,1.833443,0.595823,0.310177,0.792310,0.401979,...,1.0,1.0,-1.0,-1.0,1.0,1.0,-1.0,1.0,1.0,1.0
3,4,1,4,0.835259,0.329615,0.340366,0.216692,0.350371,0.247594,0.341194,...,-1.0,1.0,1.0,0.0,1.0,-1.0,-1.0,-1.0,-1.0,1.0
4,5,1,5,0.274185,0.061076,0.010168,0.060890,0.000000,0.213162,0.162763,...,-1.0,1.0,1.0,0.0,1.0,0.0,-1.0,-1.0,-1.0,0.0


## <a id='toc1_3_'></a>[Load target and merge](#toc0_)

In [7]:
target_df = pd.read_csv(train_target_path, sep=";")
# Some datasets ship the target file as comma-separated despite inputs using semicolons
if len(target_df.columns) == 1:
    target_df = pd.read_csv(train_target_path, sep=",")

assert "ID" in target_df.columns and "TARGET" in target_df.columns
assert target_df["ID"].is_unique
train_df = train_df.merge(target_df, on="ID", how="inner")
print("Train merged shape:", train_df.shape)
assert train_df["TARGET"].notnull().all()


Train merged shape: (636313, 112)


## <a id='toc1_4_'></a>[Feature engineering](#toc0_)

In [8]:
def add_features(df, vol_cols, ret_cols):
    X = df.copy()

    vol = X[vol_cols].astype("float32")
    ret = X[ret_cols].astype("float32")

    # Volatility stats
    X["vol_mean"] = vol.mean(axis=1)
    X["vol_std"] = vol.std(axis=1)
    X["vol_min"] = vol.min(axis=1)
    X["vol_max"] = vol.max(axis=1)
    X["vol_median"] = vol.median(axis=1)
    X["vol_q10"] = vol.quantile(0.10, axis=1)
    X["vol_q90"] = vol.quantile(0.90, axis=1)
    X["vol_first"] = vol.iloc[:, 0]
    X["vol_last"] = vol.iloc[:, -1]
    X["vol_range"] = X["vol_max"] - X["vol_min"]
    X["vol_of_vol"] = vol.diff(axis=1).abs().mean(axis=1)
    X["vol_early_late_ratio"] = (vol.iloc[:, : len(vol_cols)//2].mean(axis=1) + 1e-6) / (vol.iloc[:, len(vol_cols)//2 :].mean(axis=1) + 1e-6)

    # Intraday slope (linear fit vs index)
    idx = np.arange(len(vol_cols))
    x = idx - idx.mean()
    denom = (x**2).sum()
    X["vol_slope"] = (vol.values @ x) / denom

    # Return stats
    X["ret_sum"] = ret.sum(axis=1)
    X["ret_mean"] = ret.mean(axis=1)
    X["ret_std"] = ret.std(axis=1)
    X["ret_abs_sum"] = ret.abs().sum(axis=1)
    X["ret_pos_ratio"] = (ret > 0).mean(axis=1)
    X["ret_neg_ratio"] = (ret < 0).mean(axis=1)
    X["ret_first"] = ret.iloc[:, 0]
    X["ret_last"] = ret.iloc[:, -1]

    # Interaction features
    X["vol_mean_x_posratio"] = X["vol_mean"] * X["ret_pos_ratio"]
    X["vol_last_x_retlast"] = X["vol_last"] * X["ret_last"]

    return X


In [9]:
def add_advanced_features(df, history_df=None):
    X = df.copy()

    if history_df is None:
        history_df = X

    # Cross-day HAR-style lags per product
    agg_cols = ["vol_mean", "vol_std", "vol_last", "ret_abs_sum", "ret_mean"]
    daily = (
        history_df[["product_id", "date"] + agg_cols]
        .groupby(["product_id", "date"], as_index=False)
        .mean()
        .sort_values(["product_id", "date"])
    )

    windows = [1, 5, 10, 22, 63, 252]
    for col in agg_cols:
        for w in windows:
            feat_name = f"{col}_har_{w}"
            daily[feat_name] = (
                daily.groupby("product_id")[col]
                .transform(lambda s: s.rolling(w, min_periods=1).mean().shift(1))
            )

    # Merge HAR features back
    har_cols = [c for c in daily.columns if c not in ["product_id", "date"] + agg_cols]
    X = X.merge(daily[["product_id", "date"] + har_cols], on=["product_id", "date"], how="left")

    # Cross-sectional normalization per date (computed on X only to avoid leakage)
    cs_cols = ["vol_mean", "vol_last", "ret_abs_sum", "vol_std"]
    for col in cs_cols:
        grp = X.groupby("date")[col]
        mean = grp.transform("mean")
        std = grp.transform("std").replace(0, 1e-6)
        X[f"{col}_z"] = (X[col] - mean) / std
        X[f"{col}_rank"] = grp.rank(pct=True)

    # Market-wide features per date (computed on X only)
    for col in cs_cols:
        mean = X.groupby("date")[col].transform("mean")
        median = X.groupby("date")[col].transform("median")
        X[f"{col}_mkt_mean"] = mean
        X[f"{col}_mkt_median"] = median
        X[f"{col}_mkt_dev"] = X[col] - mean
        X[f"{col}_mkt_ratio"] = X[col] / (mean + 1e-6)

    # Cast new features to float32
    new_cols = [c for c in X.columns if c not in ["ID", "date", "product_id", "TARGET"]]
    X[new_cols] = X[new_cols].astype("float32")

    return X


## <a id='toc1_5_'></a>[Prepare datasets](#toc0_)

In [10]:
# Feature sets
raw_features = ["ID", "date", "product_id"] + vol_cols + ret_cols

# Engineered features (intraday)
train_feat = add_features(train_df[raw_features + ["TARGET"]], vol_cols, ret_cols)
test_feat = add_features(test_df[raw_features], vol_cols, ret_cols)

# Advanced features
# Train: use train history only (no transductive leakage)
train_feat = add_advanced_features(train_feat, history_df=train_feat)

# Test: use combined history to get lagged signals from train
history_feat = pd.concat([train_feat.drop(columns=["TARGET"], errors='ignore'), test_feat], axis=0, ignore_index=True)
test_feat = add_advanced_features(test_feat, history_df=history_feat)

# Separate target
y = train_feat["TARGET"].values

# Drop target from train features
X = train_feat.drop(columns=["TARGET"])

# Sanitize feature names for all models (XGB/CAT/LGB)
def sanitize_columns(cols):
    used = {}
    new_cols = []
    for c in cols:
        c2 = ''.join(ch if (ch.isalnum() or ch == '_') else '_' for ch in c)
        if not c2:
            c2 = 'f'
        if c2 in used:
            used[c2] += 1
            c2 = f"{c2}_{used[c2]}"
        else:
            used[c2] = 0
        new_cols.append(c2)
    return new_cols

sanitized = sanitize_columns(X.columns)
col_map = dict(zip(X.columns, sanitized))
X = X.rename(columns=col_map)
test_feat = test_feat.rename(columns=col_map)

print("Feature columns:", X.shape[1])


Feature columns: 188


## <a id='toc1_6_'></a>[Evaluation metric (MAPE)](#toc0_)

In [11]:
def safe_mape(y_true, y_pred, eps=1e-6):
    y_true = np.maximum(y_true, eps)
    return np.mean(np.abs(y_true - y_pred) / y_true)


## <a id='toc1_7_'></a>[Baselines](#toc0_)

In [12]:
# Global mean baseline
global_mean = np.mean(y)
base_pred = np.full_like(y, global_mean)
print("Global mean baseline MAPE:", safe_mape(y, base_pred))

# Product mean baseline
prod_mean = train_df.groupby("product_id")["TARGET"].mean()
base_pred_prod = train_df["product_id"].map(prod_mean).values
print("Product mean baseline MAPE:", safe_mape(y, base_pred_prod))


Global mean baseline MAPE: 0.6859047355824881
Product mean baseline MAPE: 0.6193279601013687


## <a id='toc1_8_'></a>[Cross-validation setup](#toc0_)

In [13]:
groups = train_df["date"].values
n_splits = 5 if RUN_MODE == "full" else 3
gkf = GroupKFold(n_splits=n_splits)


## <a id='toc1_9_'></a>[LightGBM training & evaluation](#toc0_)

In [14]:
import lightgbm as lgb

def _sanitize_feature_names(columns):
    # LightGBM rejects special JSON characters; keep alnum and underscore
    out = []
    for c in columns:
        c2 = ''.join(ch if (ch.isalnum() or ch == '_') else '_' for ch in c)
        out.append(c2 if c2 else 'f')
    return out

def _feval_mape(y_pred, dataset):
    y_true = dataset.get_label()
    eps = 1e-6
    y_true = np.maximum(y_true, eps)
    mape = np.mean(np.abs(y_true - y_pred) / y_true)
    return ('mape', mape, False)

def _feval_mape_log(y_pred, dataset):
    y_true = np.expm1(dataset.get_label())
    y_pred = np.expm1(y_pred)
    eps = 1e-6
    y_true = np.maximum(y_true, eps)
    mape = np.mean(np.abs(y_true - y_pred) / y_true)
    return ('mape', mape, False)

def train_lgbm_cv(X, y, groups, params, use_log=False):
    oof = np.zeros(len(y))
    scores = []
    feature_names = _sanitize_feature_names(X.columns)

    splits = list(gkf.split(X, y, groups))
    for fold, (tr_idx, va_idx) in enumerate(tqdm(splits, desc='CV folds')):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        if use_log:
            y_tr = np.log1p(y_tr)
            y_va_eval = y_va
            feval = _feval_mape_log
        else:
            y_va_eval = y_va
            feval = _feval_mape

        dtrain = lgb.Dataset(X_tr, label=y_tr, feature_name=feature_names)
        dvalid = lgb.Dataset(X_va, label=y_va if not use_log else np.log1p(y_va), reference=dtrain, feature_name=feature_names)

        callbacks = [
            lgb.early_stopping(200, first_metric_only=True, verbose=False),
            lgb.log_evaluation(200 if RUN_MODE == "full" else 100),
        ]

        model = lgb.train(
            params,
            dtrain,
            num_boost_round=5000 if RUN_MODE == "full" else 800,
            valid_sets=[dvalid],
            feval=feval,
            callbacks=callbacks,
        )

        preds = model.predict(X_va, num_iteration=model.best_iteration)
        if use_log:
            preds = np.expm1(preds)

        oof[va_idx] = preds
        score = safe_mape(y_va_eval, preds)
        scores.append(score)
        print(f"Fold {fold} MAPE:", score)

    print("CV MAPE mean:", np.mean(scores), "std:", np.std(scores))
    return oof, scores

# Baseline params
lgb_params = {
    "objective": "regression",
    "metric": "None",
    "learning_rate": 0.03,
    "num_leaves": 63,
    "max_depth": -1,
    "min_data_in_leaf": 40,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "lambda_l1": 0.0,
    "lambda_l2": 0.0,
    "seed": RANDOM_SEED,
    "verbose": -1,
}


In [15]:
# Train LightGBM on engineered features
lgb_oof, lgb_scores = train_lgbm_cv(X, y, groups, lgb_params, use_log=False)

# Log-transform target
lgb_oof_log, lgb_scores_log = train_lgbm_cv(X, y, groups, lgb_params, use_log=True)


CV folds:   0%|          | 0/5 [00:00<?, ?it/s]

[200]	valid_0's mape: 0.250271
[400]	valid_0's mape: 0.248172
[600]	valid_0's mape: 0.248049
[800]	valid_0's mape: 0.247683
[1000]	valid_0's mape: 0.247807


CV folds:  20%|██        | 1/5 [00:40<02:41, 40.26s/it]

Fold 0 MAPE: 0.247596233675589
[200]	valid_0's mape: 0.249407
[400]	valid_0's mape: 0.24828
[600]	valid_0's mape: 0.247907
[800]	valid_0's mape: 0.248227


CV folds:  40%|████      | 2/5 [01:11<01:45, 35.04s/it]

...

Fold 2 MAPE: 0.24410733994215067
[200]	valid_0's mape: 0.253197
[400]	valid_0's mape: 0.252313
[600]	valid_0's mape: 0.25208


CV folds:  80%|████████  | 4/5 [02:36<00:38, 38.27s/it]

Fold 3 MAPE: 0.2519109828843646
[200]	valid_0's mape: 0.25238
[400]	valid_0's mape: 0.252634


CV folds: 100%|██████████| 5/5 [02:57<00:00, 35.44s/it]

Fold 4 MAPE: 0.25178311951263466
CV MAPE mean: 0.2475350777585006 std: 0.0035339523883388003


## <a id='toc1_10_'></a>[Hyperparameter tuning with Optuna (LightGBM)](#toc0_)

In [21]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

n_trials = 50 if RUN_MODE == "full" else 10


def objective(trial):
    params = {
        "objective": "regression",
        "metric": "None",
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 31, 256),
        "max_depth": trial.suggest_int("max_depth", -1, 12),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 200),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 5.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 5.0),
        "seed": RANDOM_SEED,
        "verbose": -1,
    }

    oof, scores = train_lgbm_cv(X, y, groups, params, use_log=True)
    return np.mean(scores)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=n_trials)

print("Best params:", study.best_params)
print("Best CV MAPE:", study.best_value)


CV folds:   0%|          | 0/5 [00:00<?, ?it/s]

[200]	valid_0's mape: 0.247067
[400]	valid_0's mape: 0.244943


CV folds:  20%|██        | 1/5 [00:24<01:38, 24.52s/it]

Fold 0 MAPE: 0.24474113099466221
[200]	valid_0's mape: 0.24555
[400]	valid_0's mape: 0.242898
[600]	valid_0's mape: 0.242653
[800]	valid_0's mape: 0.242941


CV folds:  40%|████      | 2/5 [01:01<01:36, 32.01s/it]

...

Fold 2 MAPE: 0.24347578560745783
[200]	valid_0's mape: 0.251453
[400]	valid_0's mape: 0.250715


CV folds:  80%|████████  | 4/5 [03:05<00:46, 46.77s/it]

Fold 3 MAPE: 0.2504963033733325
[200]	valid_0's mape: 0.250995
[400]	valid_0's mape: 0.25119


CV folds: 100%|██████████| 5/5 [03:35<00:00, 43.19s/it]

Fold 4 MAPE: 0.25043840644695303
CV MAPE mean: 0.24657854158738823 std: 0.003276035482989103
Best params: {'learning_rate': 0.01670645341889549, 'num_leaves': 201, 'max_depth': 8, 'min_data_in_leaf': 80, 'feature_fraction': 0.8639401724742518, 'bagging_fraction': 0.6868646958618081, 'bagging_freq': 4, 'lambda_l1': 2.2375591919095026, 'lambda_l2': 2.199273654708064}
Best CV MAPE: 0.2465475001919128


In [17]:
# XGBoost and CatBoost training + OOF collection
import numpy as np

def train_xgb_cv(X, y, groups, params, use_log=False):
    import xgboost as xgb
    oof = np.zeros(len(y))
    scores = []
    splits = list(gkf.split(X, y, groups))
    for fold, (tr_idx, va_idx) in enumerate(tqdm(splits, desc='XGB CV folds')):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        if use_log:
            y_tr = np.log1p(y_tr)
            y_va_eval = y_va
        else:
            y_va_eval = y_va
        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dvalid = xgb.DMatrix(X_va, label=y_va if not use_log else np.log1p(y_va))
        model = xgb.train(
            params,
            dtrain,
            num_boost_round=3000 if RUN_MODE == 'full' else 600,
            evals=[(dvalid, 'valid')],
            early_stopping_rounds=200,
            verbose_eval=200 if RUN_MODE == 'full' else 100,
        )
        preds = model.predict(dvalid, iteration_range=(0, model.best_iteration + 1))
        if use_log:
            preds = np.expm1(preds)
        oof[va_idx] = preds
        score = safe_mape(y_va_eval, preds)
        scores.append(score)
        print(f'Fold {fold} MAPE:', score)
    print('XGB CV MAPE mean:', np.mean(scores), 'std:', np.std(scores))
    return oof, scores

def train_cat_cv(X, y, groups, params, use_log=False):
    from catboost import CatBoostRegressor
    oof = np.zeros(len(y))
    scores = []
    cat_features = [X.columns.get_loc('product_id'), X.columns.get_loc('date')]
    splits = list(gkf.split(X, y, groups))
    for fold, (tr_idx, va_idx) in enumerate(tqdm(splits, desc='CAT CV folds')):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        if use_log:
            y_tr = np.log1p(y_tr)
            y_va_eval = y_va
        else:
            y_va_eval = y_va
        model = CatBoostRegressor(**params)
        model.fit(X_tr, y_tr, cat_features=cat_features, eval_set=(X_va, y_va if not use_log else np.log1p(y_va)), verbose=200 if RUN_MODE == 'full' else 100)
        preds = model.predict(X_va)
        if use_log:
            preds = np.expm1(preds)
        oof[va_idx] = preds
        score = safe_mape(y_va_eval, preds)
        scores.append(score)
        print(f'Fold {fold} MAPE:', score)
    print('CAT CV MAPE mean:', np.mean(scores), 'std:', np.std(scores))
    return oof, scores

# Params
xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'mape',
    'eta': 0.03,
    'max_depth': 8,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': RANDOM_SEED,
}

cat_params = {
    'depth': 8,
    'learning_rate': 0.05,
    'loss_function': 'MAPE',
    'random_seed': RANDOM_SEED,
    'verbose': 200 if RUN_MODE == 'full' else 100,
}


## <a id='toc1_11_'></a>[Optional models (CatBoost / XGBoost) and ensembling](#toc0_)

In [18]:
# Optional models: run if installed
# CatBoost example:
# from catboost import CatBoostRegressor
# cat_params = {
#     "depth": 8,
#     "learning_rate": 0.05,
#     "loss_function": "MAPE",
#     "random_seed": RANDOM_SEED,
#     "verbose": 200,
# }
#
# For categorical handling, pass cat_features indices
# cat_features = [X.columns.get_loc("product_id"), X.columns.get_loc("date")]


## <a id='toc1_12_'></a>[Final training and submission](#toc0_)

In [19]:
# Ensemble blending on OOF predictions
def find_best_weights(oof_preds, y_true, step=0.05):
    n = oof_preds.shape[1]
    best = (None, 1e9)
    weights = np.arange(0, 1 + 1e-9, step)
    for w1 in weights:
        for w2 in weights:
            w3 = 1 - w1 - w2
            if w3 < 0: continue
            w = np.array([w1, w2, w3])
            preds = oof_preds @ w
            score = safe_mape(y_true, preds)
            if score < best[1]:
                best = (w, score)
    return best

# Example usage after training: oof_preds = np.vstack([lgb_oof, xgb_oof, cat_oof]).T
# best_w, best_score = find_best_weights(oof_preds, y)
# print('Best blend', best_w, best_score)


In [22]:
# Train final model on full data using best params
best_params = lgb_params.copy()
# If you ran Optuna, update best_params with study.best_params
# best_params.update(study.best_params)

# Train on full data
lgb_train = lgb.Dataset(X, label=np.log1p(y))
final_model = lgb.train(
    best_params,
    lgb_train,
    num_boost_round=2000 if RUN_MODE == "full" else 600,
    valid_sets=[lgb_train],
    verbose_eval=200,
)

# Predict test
X_test = test_feat[X.columns]
preds = final_model.predict(X_test)
preds = np.expm1(preds)

# Sanity checks
preds = np.maximum(preds, 1e-6)
assert len(preds) == len(test_df)

submission = pd.DataFrame({"ID": test_df["ID"].values, "TARGET": preds})
submission_path = Path("submission.csv")
submission.to_csv(submission_path, sep=";", index=False)

print("Saved submission to", submission_path, "rows:", len(submission))


TypeError: train() got an unexpected keyword argument 'verbose_eval'

In [ ]:
# Run All Night
metrics = []

# 1) Train LGBM base/log
lgb_oof, lgb_scores = train_lgbm_cv(X, y, groups, lgb_params, use_log=False)
metrics.append({'model': 'lgb_base', 'mape_mean': float(np.mean(lgb_scores)), 'mape_std': float(np.std(lgb_scores))})

lgb_oof_log, lgb_scores_log = train_lgbm_cv(X, y, groups, lgb_params, use_log=True)
metrics.append({'model': 'lgb_log', 'mape_mean': float(np.mean(lgb_scores_log)), 'mape_std': float(np.std(lgb_scores_log))})

# 2) Optuna tuning (optional)
best_params = lgb_params.copy()
if 'study' in locals():
    best_params.update(study.best_params)

# 3) Train XGB and CAT (original scale for stable MAPE)
xgb_oof, xgb_scores = train_xgb_cv(X, y, groups, xgb_params, use_log=False)
metrics.append({'model': 'xgb', 'mape_mean': float(np.mean(xgb_scores)), 'mape_std': float(np.std(xgb_scores))})

cat_oof, cat_scores = train_cat_cv(X, y, groups, cat_params, use_log=False)
metrics.append({'model': 'cat', 'mape_mean': float(np.mean(cat_scores)), 'mape_std': float(np.std(cat_scores))})

# Save OOF preds
np.save(RUN_DIR / 'oof_lgb.npy', lgb_oof)
np.save(RUN_DIR / 'oof_lgb_log.npy', lgb_oof_log)
np.save(RUN_DIR / 'oof_xgb.npy', xgb_oof)
np.save(RUN_DIR / 'oof_cat.npy', cat_oof)

# 4) Blend (use LGB log + XGB + CAT)
oof_preds = np.vstack([lgb_oof_log, xgb_oof, cat_oof]).T
step = 0.05 if RUN_MODE == 'full' else 0.1
best_w, best_score = find_best_weights(oof_preds, y, step=step)
print('Best blend weights', best_w, 'MAPE', best_score)
(RUN_DIR / 'best_blend.txt').write_text(f'{best_w} {best_score}')
metrics.append({'model': 'blend', 'mape_mean': float(best_score), 'mape_std': 0.0})

# Save metrics
pd.DataFrame(metrics).to_csv(RUN_DIR / 'metrics.csv', index=False)

# 5) Train final models on full data and create submission
# LightGBM full
lgb_train = lgb.Dataset(X, label=np.log1p(y))
final_lgb = lgb.train(best_params, lgb_train, num_boost_round=2000 if RUN_MODE == 'full' else 600)

# XGB full
import xgboost as xgb
dtrain_full = xgb.DMatrix(X, label=y)
final_xgb = xgb.train(xgb_params, dtrain_full, num_boost_round=2000 if RUN_MODE == 'full' else 600)

# CAT full
from catboost import CatBoostRegressor
cat_features = [X.columns.get_loc('product_id'), X.columns.get_loc('date')]
final_cat = CatBoostRegressor(**cat_params)
final_cat.fit(X, y, cat_features=cat_features, verbose=200 if RUN_MODE == 'full' else 100)

# Predictions
X_test = test_feat[X.columns]
pred_lgb = np.expm1(final_lgb.predict(X_test))
pred_xgb = final_xgb.predict(xgb.DMatrix(X_test))
pred_cat = final_cat.predict(X_test)

preds = best_w[0] * pred_lgb + best_w[1] * pred_xgb + best_w[2] * pred_cat
preds = np.maximum(preds, 1e-6)

submission = pd.DataFrame({'ID': test_df['ID'].values, 'TARGET': preds})
submission.to_csv('submission.csv', sep=';', index=False)
submission.to_csv(RUN_DIR / 'submission.csv', sep=';', index=False)
print('Saved submission')
